# NMF on ModulAir 00683

In [13]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [18]:
#importing data from Modulair MOD-00068
data = pd.read_csv('MOD-00683.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-29T16:55:02Z,576028598,2025-12-29T11:55:02Z,MOD-00683,96.3,8.8,42.941,17.835,6.328,1.280,0.876,...,NaN,NaN,16587,16588,16589,NaN,NaN,NaN,NaN,7.13
2025-12-29T16:54:02Z,576028597,2025-12-29T11:54:02Z,MOD-00683,96.0,8.8,41.879,16.737,5.744,1.112,0.770,...,NaN,NaN,16587,16588,16589,NaN,NaN,NaN,NaN,6.67
2025-12-29T16:53:02Z,576028596,2025-12-29T11:53:02Z,MOD-00683,95.6,8.8,42.907,16.337,6.045,1.202,0.752,...,NaN,NaN,16587,16588,16589,NaN,NaN,NaN,NaN,5.51
2025-12-29T16:35:06Z,576019139,2025-12-29T11:35:06Z,MOD-00683,94.9,8.6,36.847,14.615,5.084,1.031,0.690,...,NaN,NaN,16587,16588,16589,NaN,NaN,NaN,NaN,4.88
2025-12-29T16:34:06Z,576019142,2025-12-29T11:34:06Z,MOD-00683,94.7,8.5,35.261,13.451,5.089,0.894,0.498,...,NaN,NaN,16587,16588,16589,NaN,NaN,NaN,NaN,6.94


In [19]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-29T16:55:02Z,2025-12-29T11:55:02Z,NaN,NaN,NaN,NaN,42.941,17.835,6.328,1.280,0.876,0.303
2025-12-29T16:54:02Z,2025-12-29T11:54:02Z,NaN,NaN,NaN,NaN,41.879,16.737,5.744,1.112,0.770,0.244
2025-12-29T16:53:02Z,2025-12-29T11:53:02Z,NaN,NaN,NaN,NaN,42.907,16.337,6.045,1.202,0.752,0.219
2025-12-29T16:35:06Z,2025-12-29T11:35:06Z,NaN,NaN,NaN,NaN,36.847,14.615,5.084,1.031,0.690,0.284
2025-12-29T16:34:06Z,2025-12-29T11:34:06Z,NaN,NaN,NaN,NaN,35.261,13.451,5.089,0.894,0.498,0.182


In [20]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29T11:55:02Z,NaN,NaN,NaN,NaN,42.941,17.835,6.328,1.280,0.876,0.303
1,2025-12-29T11:54:02Z,NaN,NaN,NaN,NaN,41.879,16.737,5.744,1.112,0.770,0.244
2,2025-12-29T11:53:02Z,NaN,NaN,NaN,NaN,42.907,16.337,6.045,1.202,0.752,0.219
3,2025-12-29T11:35:06Z,NaN,NaN,NaN,NaN,36.847,14.615,5.084,1.031,0.690,0.284
4,2025-12-29T11:34:06Z,NaN,NaN,NaN,NaN,35.261,13.451,5.089,0.894,0.498,0.182


In [21]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'], 
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-29 11:55:02,NaN,NaN,NaN,NaN,42.941,17.835,6.328,1.280,0.876,0.303
1,2025-12-29 11:54:02,NaN,NaN,NaN,NaN,41.879,16.737,5.744,1.112,0.770,0.244
2,2025-12-29 11:53:02,NaN,NaN,NaN,NaN,42.907,16.337,6.045,1.202,0.752,0.219
3,2025-12-29 11:35:06,NaN,NaN,NaN,NaN,36.847,14.615,5.084,1.031,0.690,0.284
4,2025-12-29 11:34:06,NaN,NaN,NaN,NaN,35.261,13.451,5.089,0.894,0.498,0.182


In [22]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

C:\Users\alexa\AppData\Local\Temp\ipykernel_12448\2364822326.py:13: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  data = data.round(decimals = 2)


,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
2,2025-04-15 12:00:00,1040.32,26.69,69.60,2.45,8.52,0.80,0.27,0.07,0.10,0.07
3,2025-04-15 13:00:00,948.32,22.58,67.31,2.93,5.22,0.52,0.19,0.05,0.08,0.06
4,2025-04-15 14:00:00,782.33,7.89,66.14,2.98,2.79,0.35,0.15,0.05,0.08,0.06
5,2025-04-15 15:00:00,722.06,8.39,66.34,2.85,1.80,0.29,0.13,0.04,0.08,0.06
6,2025-04-15 16:00:00,703.11,14.13,63.22,2.02,1.49,0.24,0.12,0.04,0.07,0.05
...,...,...,...,...,...,...,...,...,...,...,...
5983,2025-12-27 13:00:00,960.70,27.81,31.29,1.63,4.81,0.74,0.27,0.06,0.07,0.05
5984,2025-12-27 14:00:00,1039.95,27.45,31.30,1.73,4.97,0.68,0.21,0.05,0.06,0.04
5985,2025-12-27 15:00:00,1080.72,28.23,30.12,1.75,4.86,0.67,0.23,0.05,0.06,0.04
5986,2025-12-27 16:00:00,1040.30,29.39,28.21,1.91,5.11,0.68,0.23,0.05,0.06,0.03


In [23]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 12:00:00,1040.32,26.69,69.60,2.45,8.52,0.80,0.27,0.07,0.10,0.07
2025-04-15 13:00:00,948.32,22.58,67.31,2.93,5.22,0.52,0.19,0.05,0.08,0.06
2025-04-15 14:00:00,782.33,7.89,66.14,2.98,2.79,0.35,0.15,0.05,0.08,0.06
2025-04-15 15:00:00,722.06,8.39,66.34,2.85,1.80,0.29,0.13,0.04,0.08,0.06
2025-04-15 16:00:00,703.11,14.13,63.22,2.02,1.49,0.24,0.12,0.04,0.07,0.05


In [24]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [25]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled
   
# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [26]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 12:00:00,0.473146,0.581735,0.753736,0.041589,0.103624,0.025592,0.018170,0.018617,0.037037,0.041420
2025-04-15 13:00:00,0.431304,0.492153,0.728937,0.049737,0.063488,0.016635,0.012786,0.013298,0.029630,0.035503
2025-04-15 14:00:00,0.355810,0.171970,0.716266,0.050586,0.033933,0.011196,0.010094,0.013298,0.029630,0.035503
2025-04-15 15:00:00,0.328399,0.182868,0.718432,0.048379,0.021892,0.009277,0.008748,0.010638,0.029630,0.035503
2025-04-15 16:00:00,0.319780,0.307977,0.684644,0.034290,0.018122,0.007678,0.008075,0.010638,0.025926,0.029586


In [31]:
data.to_csv(r'C:\Users\alexa\Jupyter Notebook\MOD-00068_timeseries_hourly_scaled.csv')

## Implementing NMF

In [32]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [33]:
W = nmf.fit_transform(data) 
H = nmf.components_   
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-04-15 12:00:00,0.497890,0.573332,0.742988,0.057867,0.099060,0.024170,0.014370,0.013320,0.034072,0.046820
2025-04-15 13:00:00,0.451994,0.485472,0.720033,0.053274,0.062517,0.008917,0.007510,0.009124,0.029840,0.042653
2025-04-15 14:00:00,0.350677,0.173854,0.716858,0.042140,0.054540,0.006549,0.007093,0.009306,0.030646,0.043176
2025-04-15 15:00:00,0.347716,0.176972,0.708231,0.041772,0.053884,0.005154,0.005581,0.007623,0.028097,0.040471
2025-04-15 16:00:00,0.365198,0.293292,0.662965,0.043510,0.050440,0.002826,0.003061,0.004798,0.022994,0.034706
...,...,...,...,...,...,...,...,...,...,...
2025-12-27 13:00:00,0.363043,0.628991,0.366973,0.041167,0.073766,0.028700,0.018355,0.016662,0.029118,0.035472
2025-12-27 14:00:00,0.369305,0.630581,0.378793,0.041790,0.081599,0.028901,0.016649,0.014117,0.025464,0.031884
2025-12-27 15:00:00,0.373722,0.652038,0.371546,0.042215,0.082992,0.030187,0.017501,0.014858,0.026182,0.032476


In [34]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.076130,0.011327,0.072640,0.008059
1,0.074788,0.002060,0.059946,0.005969
2,0.074691,0.000000,0.012826,0.006523
3,0.073792,0.000000,0.013462,0.005133
4,0.069075,0.000000,0.031942,0.002815
...,...,...,...,...
5816,0.036852,0.012209,0.088146,0.011322
5817,0.037874,0.014056,0.088127,0.008912
5818,0.037061,0.014574,0.091505,0.009460
5819,0.034260,0.015173,0.095149,0.008179


In [35]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [36]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.311639,1.194276,9.597695,0.521501,0.730216,0.000000,0.000000,0.019455,0.265159,0.430791
Feature 2,1.408395,0.267948,1.087565,0.095483,3.837689,1.419615,0.495100,0.176192,0.043484,0.000000
Feature 3,2.083199,6.599175,0.000000,0.231425,0.000000,0.000000,0.000000,0.002494,0.000000,0.007655
Feature 4,0.294049,0.001553,0.000000,0.033855,0.000000,1.003964,1.087277,1.198878,1.661848,1.671244


In [37]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-04-15 12:00:00,0.076130,0.011327,0.072640,0.008059
2025-04-15 13:00:00,0.074788,0.002060,0.059946,0.005969
2025-04-15 14:00:00,0.074691,0.000000,0.012826,0.006523
2025-04-15 15:00:00,0.073792,0.000000,0.013462,0.005133
2025-04-15 16:00:00,0.069075,0.000000,0.031942,0.002815
...,...,...,...,...
2025-12-27 13:00:00,0.036852,0.012209,0.088146,0.011322
2025-12-27 14:00:00,0.037874,0.014056,0.088127,0.008912
2025-12-27 15:00:00,0.037061,0.014574,0.091505,0.009460


In [38]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.311639,1.194276,9.597695,0.521501,0.730216,0.000000,0.000000,0.019455,0.265159,0.430791
Factor 2,1.408395,0.267948,1.087565,0.095483,3.837689,1.419615,0.495100,0.176192,0.043484,0.000000
Factor 3,2.083199,6.599175,0.000000,0.231425,0.000000,0.000000,0.000000,0.002494,0.000000,0.007655
Factor 4,0.294049,0.001553,0.000000,0.033855,0.000000,1.003964,1.087277,1.198878,1.661848,1.671244


In [39]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.527110,0.114870,0.324523,0.014536,0.018961
no,0.573141,0.070009,0.324095,0.015045,0.017710
no2,0.122704,0.018366,0.863970,0.000065,-0.005104
o3,0.935411,0.070715,0.000000,0.000000,-0.006127
bin0,0.224225,0.786187,0.000000,0.000000,-0.010412
bin1,0.000000,0.784441,0.000000,0.336230,-0.120671
bin2,0.000000,0.464087,0.000000,0.617696,-0.081783
bin3,0.031467,0.190121,0.005140,0.784054,-0.010781
bin4,0.276308,0.030230,0.000000,0.700213,-0.006751
bin5,0.387440,0.000000,0.008773,0.607755,-0.003968


In [41]:
results.to_csv(r'C:/Users/alexa/Jupyter Notebook/4_factor_results.csv')
comp.to_csv(r'C:/Users/alexa/Jupyter Notebook/4_factor_comp.csv')
res.to_csv(r'C:/Users/alexa/Jupyter Notebook/4_factor_resid.csv')